# modeling_v13 — Ablation: core10 제거 (정직축 GKF 전용)

**목적**: 확정 셋 **Conservative-GA(99)** 에서 **core10(레짐/시간 백본 10개)만 제거**한 **89 피처** 를
같은 조건으로 GKF(C20) 재측정 → **core10 이 정직축에 실제 얼마나 기여하는지**를 숫자로 확인.

> ⚠️ **진단용 ablation — 게이트·의사결정 불변.** 확정 챔피언 파이프라인은 **core10 포함 99 그대로**다.
> 이 노트북은 core10 의 정직축 기여를 정량화할 뿐, 어떤 게이트/셋도 바꾸지 않는다(R1).

**비교 구성** (2셋 × 2모델 = 4 구성, **GKF 5-fold OOF 만**):
- `full99` (core10 포함) — 재현 self-check 겸 기준
- `no_core10_89` (core10 제거) — champion 5 + GA 선별 84 = 89

---
### 전제 파일 (노트북 폴더 또는 `data/`·`colab_GA/`)
- `v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `feature_diet_selected.json` · `select_result_Conservative_GA.json`

### 런타임 · 실행
- **Restart & Run All**. GKF only = 4 구성 × 5 fold = **20 fit**. 로컬 ~20분(LGBM 10 fit + XGB 10 fit) · CPU. 빠른 점검: `QUICK=True`(env `M5_QUICK=1`).
- XGB Colab GPU: `XGB_DEVICE="cuda"`(수치는 로컬 CPU 재확인 = 공식, R6).

### 확인 포인트 (실행 후 회신)
- **재현 self-check**: `full99` LGBM GKF ≈ **71.366**(Δ<0.05) · XGB ≈ **70.773**(Δ<0.3, xgboost 버전 의존) → 환경 무결.
- `no_core10_89` 의 LGBM·XGB GKF 및 **Δ(무 − 유)** — core10 기여폭.
- 저장: `ablation_no_core10_results.{json,csv}`.

> **정직 유의**: core10 은 레짐/시간(dominant 신호)을 담는다. 제거 시 GKF 는 **악화가 예상**되나 폭은 미지 — **돌려서 확인**. 필수 5센서 floor 는 champion 5 로 core10 없이도 유지된다(assert).


In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# 동결 상수
ID_COL, TARGET_COL = "C64", "C65"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33",
          "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SIGMA_C65 = 261.7

# 모델 (미튜닝 baseline — 기존 동결 수치와 apples-to-apples)
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BEST_ROUNDS = 705
XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")
XGB_PARAMS = dict(n_estimators=BEST_ROUNDS, learning_rate=0.029, max_depth=8,
                  subsample=0.86, colsample_bytree=0.63, min_child_weight=5,
                  reg_lambda=0.02, reg_alpha=5.0, gamma=0.0,
                  tree_method="hist", device=XGB_DEVICE, random_state=42, n_jobs=-1, verbosity=0)

# 동결 참조값 (로컬 venv 산; R6)
REF_FULL = {"LGBM": 71.366, "XGBoost": 70.773}   # full99 재현 self-check 기준
CORE10_ONLY_GKF = 78.181                          # M2: core10 단독 GKF
B0_REF, ANCHOR = 71.212, 70.712

QUICK = bool(int(os.environ.get("M5_QUICK", "0")))
if QUICK:
    BEST_ROUNDS = 60; XGB_PARAMS["n_estimators"] = 60
    print("[QUICK] rounds 60 스모크")

def _find(name):
    for d in [".", "data", "colab_GA", "..",
              os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"),
              os.path.join("..", "modeling_v13", "colab_GA")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 - 노트북 폴더(또는 data/·colab_GA/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz")
META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json")
SEL_CONS = _find("select_result_Conservative_GA.json")
print("파일 확인:", *[os.path.basename(x) for x in [POOL, META, DIET, SEL_CONS]])
print("xgboost", xgb.__version__, "| lightgbm", lgb.__version__)


파일 확인: v13_fdc_pool_wf_oof.csv.gz core10_meta_wf.csv feature_diet_selected.json select_result_Conservative_GA.json
xgboost 3.3.0 | lightgbm 4.6.0


In [2]:
# 헬퍼
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

# v2.2 환경독립 GroupKFold (동결 stable 폴드맵)
def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fold_sizes = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fold_sizes)); fold_sizes[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def stable_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

def oof_gkf_lgb(splits, df, feats, y):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = lgb.train(M8_PARAMS, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=BEST_ROUNDS)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def oof_gkf_xgb(splits, df, feats, y):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = xgb.XGBRegressor(**XGB_PARAMS).fit(df.iloc[tr][feats], y[tr])
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)


In [3]:
# 로드 & 셋 구성: full99 vs no_core10_89
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
y = df[TARGET_COL].to_numpy(float)
groups = df["C20"].to_numpy()
diet = json.loads(open(DIET, encoding="utf-8").read())
champions = list(diet["champions"]["Conservative"].values())
fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
selp = json.loads(open(SEL_CONS, encoding="utf-8").read())["selected_features"]

full99 = list(dict.fromkeys(fixed + selp))
core10_in = [f for f in CORE10 if f in df.columns]
no_core10 = [f for f in full99 if f not in set(CORE10)]     # core10 제거

SETS = {"full99": full99, "no_core10_89": no_core10}
for name, fs in SETS.items():
    ok, have = floor_ok(fs)
    assert all(f in df.columns for f in fs), f"{name}: 누락 피처"
    assert "C20" not in fs and "C64" not in fs and "fold_kf5" not in fs, f"{name}: 누수 피처!"
    assert ok, f"{name}: 필수 5센서 floor 위반 {have}"        # R10 (champion 이 core10 없이도 보장)
    print(f"{name:14s} n={len(fs):3d}  floor={have}")
assert len(full99) == 99 and len(no_core10) == 99 - len(core10_in), "셋 크기 이상"
print(f"제거된 core10({len(core10_in)}): {core10_in}")
STABLE_SPLITS = stable_splits(groups)
print(f"df {df.shape} | unique C20 {len(np.unique(groups))} lot | CV=stable GKF 5-fold")


full99         n= 99  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
no_core10_89   n= 89  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
제거된 core10(10): ['is_high_regime', 'high_regime_days', 'days_since_last_pm', 'C33', 'dslp_x_hour', 'hour', 'hour_x_c33', 'C60_mean_step4', 'C59_mean_step4', 'is_special_recipe']
df (11939, 667) | unique C20 1217 lot | CV=stable GKF 5-fold


In [4]:
# GKF 실행: 2셋 × 2모델 = 20 fit
rows = []
for name, fs in SETS.items():
    for model, fn in [("LGBM", oof_gkf_lgb), ("XGBoost", oof_gkf_xgb)]:
        t = time.time()
        gk = fn(STABLE_SPLITS, df, fs, y)
        rows.append(dict(feature_set=name, model=model, n_feats=len(fs),
                         GroupKFold_C20=round(gk, 3), R2=r2_honest(gk), sec=round(time.time()-t, 0)))
        print(f"  {name:14s} {model:8s} GKF={gk:.3f}  R2={r2_honest(gk)}  ({time.time()-t:.0f}s)")

res = pd.DataFrame(rows)
# 재현 self-check
print("\n[재현 self-check] full99 (동결값 대비)")
for model, ref in REF_FULL.items():
    g = float(res[(res.feature_set=="full99") & (res.model==model)].GroupKFold_C20.iloc[0])
    tol = 0.05 if model == "LGBM" else 0.30   # XGB 는 버전 의존(느슨)
    print(f"  {model:8s} full99 GKF={g:.3f}  (동결 {ref}, Δ{g-ref:+.3f})  "
          f"{'OK' if abs(g-ref) < tol else '주의 - 환경/핀 점검'}")
res


  full99         LGBM     GKF=71.366  R2=0.9256  (244s)
  full99         XGBoost  GKF=70.517  R2=0.9274  (44s)
  no_core10_89   LGBM     GKF=79.295  R2=0.9082  (232s)
  no_core10_89   XGBoost  GKF=78.922  R2=0.9091  (59s)

[재현 self-check] full99 (동결값 대비)
  LGBM     full99 GKF=71.366  (동결 71.366, Δ+0.000)  OK
  XGBoost  full99 GKF=70.517  (동결 70.773, Δ-0.256)  OK


,feature_set,model,n_feats,GroupKFold_C20,R2,sec
0,full99,LGBM,99,71.366,0.9256,244.0
1,full99,XGBoost,99,70.517,0.9274,44.0
2,no_core10_89,LGBM,89,79.295,0.9082,232.0
3,no_core10_89,XGBoost,89,78.922,0.9091,59.0


In [5]:
# 요약: core10 기여(무 − 유) + 저장
def gkf(name, model):
    return float(res[(res.feature_set==name) & (res.model==model)].GroupKFold_C20.iloc[0])

summary = {}
print("=== core10 기여 (Δ = no_core10 − full99; +면 제거 시 악화) ===")
for model in ["LGBM", "XGBoost"]:
    full, abl = gkf("full99", model), gkf("no_core10_89", model)
    d = abl - full
    summary[model] = dict(full99=round(full, 3), no_core10_89=round(abl, 3),
                          delta_no_minus_full=round(d, 3),
                          R2_full=r2_honest(full), R2_no_core10=r2_honest(abl))
    print(f"  {model:8s}: full99 {full:.3f} → no_core10 {abl:.3f}   Δ {d:+.3f}   "
          f"(R2 {r2_honest(full)} → {r2_honest(abl)})")
print(f"\n  참고: core10 단독 GKF={CORE10_ONLY_GKF} (M2) · 결합 B0={B0_REF} · 앵커={ANCHOR}")
print("  ※ 진단용 — 게이트·확정 셋 불변. 확정 챔피언은 core10 포함 99 그대로.")

out = dict(purpose="core10 ablation (GKF only)", cv="stable_group_kfold v2.2",
           results=rows, summary=summary,
           ref=dict(full_frozen=REF_FULL, core10_only_gkf=CORE10_ONLY_GKF, B0=B0_REF, anchor=ANCHOR),
           note="진단 ablation. 확정 파이프라인은 core10 포함 그대로(게이트 불변, R1). "
                "XGB 절대값 버전 의존(R6). floor 는 champion 으로 core10 없이도 유지.")
json.dump(out, open("ablation_no_core10_results.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
res.to_csv("ablation_no_core10_results.csv", index=False)
print("saved: ablation_no_core10_results.json / ablation_no_core10_results.csv")


=== core10 기여 (Δ = no_core10 − full99; +면 제거 시 악화) ===
  LGBM    : full99 71.366 → no_core10 79.295   Δ +7.929   (R2 0.9256 → 0.9082)
  XGBoost : full99 70.517 → no_core10 78.922   Δ +8.405   (R2 0.9274 → 0.9091)

  참고: core10 단독 GKF=78.181 (M2) · 결합 B0=71.212 · 앵커=70.712
  ※ 진단용 — 게이트·확정 셋 불변. 확정 챔피언은 core10 포함 99 그대로.
saved: ablation_no_core10_results.json / ablation_no_core10_results.csv


---
### 해석 가이드
- **Δ(no_core10 − full99) 클수록** = core10(레짐/시간)의 정직축 기여가 큼. M2 근거(core10 단독 78.2 vs 결합 71.2)상 **큰 악화 예상**.
- `no_core10_89` 가 core10 단독(78.2)보다도 나쁘면 → **센서만으론 레짐 신호를 못 대신함**(레짐 지배 재확인). 78.2보다 나으면 → 센서가 레짐 정보를 부분적으로 품고 있다는 뜻.
- **결론과 무관하게 확정 모델은 core10 포함 99 유지** — 이 실험은 "왜 core10 이 필요한가"의 정량 근거일 뿐, 게이트/셋 변경 아님.

> 로컬 venv 수치가 공식(R6). Colab/미러 수치는 상대비교 전용. XGB 는 로컬 xgboost 버전 핀 확인(§10.6 P0-3).
